In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [ ]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

In [ ]:
from pyspark.sql import functions as F

# Define the mapping of old labcode values to new labcode values
labcode_mapping = {
    "ABSOLUTE GRANU": "ABSOLUTE_GRANU",
    "ABSOLUTE LYMPH": "ABSOLUTE_LYMPH",
    "ANA WITH REFLE": "ANA_WITH_REFLE",
    "Glucose 3 Hour Specimen": "Glucose_3_Hour_Specimen",
    "HEPATITIS C AN": "HEPATITIS_C_AN",
    "IRON BINDING C": "IRON_BINDING_C",
    "SEX HORMONE BI": "SEX_HORMONE_BI",
    "TESTOSTERONE F": "TESTOSTERONE_F",
    "TSH WITH REFLE": "TSH_WITH_REFLE"
}

# Apply the renaming transformation to the labcode column
Epilepsy_Control_Lab = Epilepsy_Control_Lab.withColumn(
    "labcode",
    F.when(F.col("labcode") == "ABSOLUTE GRANU", "ABSOLUTE_GRANU")
    .when(F.col("labcode") == "ABSOLUTE LYMPH", "ABSOLUTE_LYMPH")
    .when(F.col("labcode") == "ANA WITH REFLE", "ANA_WITH_REFLE")
    .when(F.col("labcode") == "Glucose 3 Hour Specimen", "Glucose_3_Hour_Specimen")
    .when(F.col("labcode") == "HEPATITIS C AN", "HEPATITIS_C_AN")
    .when(F.col("labcode") == "IRON BINDING C", "IRON_BINDING_C")
    .when(F.col("labcode") == "SEX HORMONE BI", "SEX_HORMONE_BI")
    .when(F.col("labcode") == "TESTOSTERONE F", "TESTOSTERONE_F")
    .when(F.col("labcode") == "TSH WITH REFLE", "TSH_WITH_REFLE")
    .otherwise(F.col("labcode"))
)

# Show the result
Epilepsy_Control_Lab.show(truncate=False)

In [ ]:
# Filter Epilepsy_Control_Lab to contain only rows with labcode value "TSH_WITH_REFLE"
filtered_df = Epilepsy_Control_Lab.filter(Epilepsy_Control_Lab["labcode"] == "TSH_WITH_REFLE")

# Show the filtered DataFrame
filtered_df.show(truncate=False)

In [ ]:
print(Epilepsy_Control_Lab.count())

In [ ]:
print(Epilepsy_Control_Lab.select("personid").distinct().count())

In [ ]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = Epilepsy_Control_Lab.rdd.getNumPartitions()
Epilepsy_Control_Lab_repart = Epilepsy_Control_Lab.repartition(reparNum)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define a window function to partition data into groups of 45000 unique personids
window_spec = Window.orderBy('personid').rowsBetween(Window.unboundedPreceding, 45000)

# Assign a group number to each unique personid
Epilepsy_Control_Lab_repart_with_group = (
    Epilepsy_Control_Lab_repart
    .select('personid')
    .distinct()
    .withColumn('group', F.expr('FLOOR((row_number() OVER (ORDER BY personid) - 1) / 45000)'))
)

# Join the group information back to the original DataFrame
Epilepsy_Control_Lab_repart_with_group = (
    Epilepsy_Control_Lab_repart
    .join(Epilepsy_Control_Lab_repart_with_group, on='personid', how='inner')
)

# Pivot and aggregate separately for each group and write to Parquet files
for i in range(9):
    group_df = (
        Epilepsy_Control_Lab_repart_with_group
        .filter(F.col('group') == i)
        .groupby('personid')
        .pivot('labcode')
        .agg(F.first('New_updated_Interpretation'))
    )
    # Write to Parquet file
    group_df.write.parquet(f'file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_{i}_lab_data.parquet')

In [ ]:
Epilepsy_Control_group_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_0_lab_data.parquet")

In [ ]:
# Get the list of column names
column_names = Epilepsy_Control_group_Lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
Epilepsy_Control_group_Lab.printSchema()

In [ ]:
print(Epilepsy_Control_group_Lab.count())

In [ ]:
Epilepsy_Control_group_Lab1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_1_lab_data.parquet")

In [ ]:
# Get the list of column names
column_names = Epilepsy_Control_group_Lab1.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
print(Epilepsy_Control_group_Lab1.count())

In [ ]:
Epilepsy_Control_group_Lab2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_2_lab_data.parquet")

In [ ]:
print(Epilepsy_Control_group_Lab2.count())

In [ ]:
Epilepsy_Control_group_Lab1.printSchema()

In [ ]:
Epilepsy_Control_group8_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_8_lab_data.parquet")

In [ ]:
print(Epilepsy_Control_group8_Lab.count())

In [ ]:
# Get the list of column names
column_names = Epilepsy_Control_group8_Lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
from functools import reduce
from pyspark.sql.functions import broadcast

# Define the file paths for the nine Parquet files
file_paths = [
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_0_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_1_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_2_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_3_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_4_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_5_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_6_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_7_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_8_lab_data.parquet"
]

# Read the Parquet files into DataFrames
dfs = [spark.read.parquet(file_path) for file_path in file_paths]

# Define a function to join two DataFrames with shuffle hint
def join_with_shuffle(df1, df2):
    return df1.join(broadcast(df2.hint("shuffle")), "personid", "outer")

# Perform the initial join
combined_df = dfs[0]

# Loop through the remaining DataFrames and perform the outer join
for i in range(1, len(dfs)):
    combined_df = join_with_shuffle(combined_df, dfs[i])

# Show the combined table
combined_df.show(truncate=False)
# from functools import reduce
# from pyspark.sql.functions import broadcast

# # Define the file paths for the nine Parquet files
# file_paths = [
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_0_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_1_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_2_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_3_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_4_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_5_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_6_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_7_lab_data.parquet",
#     "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_8_lab_data.parquet"
# ]

# # Read the Parquet files into DataFrames
# dfs = [spark.read.parquet(file_path) for file_path in file_paths]

# # Define a function to join two DataFrames with shuffle hint
# def join_with_shuffle(df1, df2):
#     return df1.join(broadcast(df2.hint("shuffle")), "personid", "outer")

# # Perform full outer join on the nine tables using shuffle hint
# combined_df = reduce(join_with_shuffle, dfs)

# # Show the combined table
# combined_df.show(truncate=False)

In [ ]:
# Define the list of Parquet files
parquet_files = [
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_0_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_1_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_2_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_3_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_4_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_5_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_6_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_7_lab_data.parquet",
    "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/group_8_lab_data.parquet"
]

# Initialize an empty DataFrame
stacked_df = None

# Read each Parquet file individually and stack up the DataFrames
for file in parquet_files:
    df = spark.read.parquet(file)
    if stacked_df is None:
        stacked_df = df
    else:
        stacked_df = stacked_df.union(df)

# Write the stacked DataFrame to a new Parquet file
stacked_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_lab_group_data.parquet")

In [ ]:
from pyspark.sql import functions as F

# Get the first 100,000 unique personid
unique_personids = Epilepsy_Control_Lab_repart.select('personid').distinct().limit(50000)

# Filter the DataFrame to include only those 100,000 unique personid
filtered_df = Epilepsy_Control_Lab_repart.join(unique_personids, on='personid', how='inner')

# Perform the pivot operation
pivoted_lab_df1 = (
    filtered_df
    .groupby('personid')
    .pivot('labcode')
    .agg(F.first('New_updated_Interpretation'))
)

In [ ]:

# Pivot the labcode column values into individual columns and repartition
pivoted_lab_df = (
    Epilepsy_Control_Lab_repart.groupby('personid')
    .pivot('labcode')
    .agg(F.first('New_updated_Interpretation'))
)
# pivoted_lab_df = pivoted_lab_df.withColumnRenamed('personid', 'pivoted_personid')

In [ ]:
# Rename columns
renamed_df = pivoted_lab_df1.withColumnRenamed("ABSOLUTE GRANU", "ABSOLUTE_GRANU") \
                            .withColumnRenamed("ABSOLUTE LYMPH", "ABSOLUTE_LYMPH") \
                            .withColumnRenamed("ANA WITH REFLE", "ANA_WITH_REFLE") \
                            .withColumnRenamed("Glucose 3 Hour Specimen", "Glucose_3_Hour_Specimen") \
                            .withColumnRenamed("HEPATITIS C AN", "HEPATITIS_C_AN") \
                            .withColumnRenamed("IRON BINDING C", "IRON_BINDING_C") \
                            .withColumnRenamed("SEX HORMONE BI", "SEX_HORMONE_BI") \
                            .withColumnRenamed("TESTOSTERONE F", "TESTOSTERONE_F") \
                            .withColumnRenamed("TSH WITH REFLE", "TSH_WITH_REFLE")

In [ ]:
reparNum1 = renamed_df.rdd.getNumPartitions()
print(reparNum1)

In [ ]:
renamed_df_repart = renamed_df.repartition(reparNum1)

In [ ]:
renamed_df_repart.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_pivoted_Control_Part0')

In [ ]:
print(renamed_df_repart.count())

In [ ]:
# Get the list of column names
column_names = renamed_df_repart.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [2]:
Epilepsy_Cohort_SuperSet = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoin")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
print(Epilepsy_Cohort_SuperSet.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

118524


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Get the list of column names
column_names = Epilepsy_Cohort_SuperSet.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
spark.conf.set("spark.sql.join.preferSortMergeJoin","false")

In [ ]:
Cohort_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Cohort_S3")

In [ ]:
from pyspark.sql.functions import col, sum as _sum, when

# Calculate the total number of rows
total_count = Cohort_Commo_Med.count()

# Create expressions to count nulls for each column
null_counts = [(_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)) for c in Cohort_Commo_Med.columns]

# Apply the expressions to the DataFrame
null_counts_df = Cohort_Commo_Med.select(*null_counts)

# Collect the results
null_counts_row = null_counts_df.collect()[0].asDict()

# Calculate the percentage of null values
empty_values_stats = [(col, null_counts_row[col], (null_counts_row[col] / total_count) * 100) for col in Cohort_Commo_Med.columns]

# Convert the results to a Spark DataFrame
empty_values_df = spark.createDataFrame(empty_values_stats, ["Column", "EmptyCount", "EmptyPercentage"])

# Show the results
empty_values_df.show(truncate=False)

In [ ]:
# Cohort_Commo_Med.printSchema()
# Get the list of column names
column_names = Cohort_Commo_Med.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_pivoted_Cohort_Lab2")

In [ ]:
lab.printSchema()

In [ ]:
print(lab.count())

In [ ]:
# Get the list of column names
column_names = lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
import math

def split_df(df, num_split):
    total_columns = len(df.columns)
    first_column = df.columns[0]  # Get the name of the first column
    
    splitted = []
    
    num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
    for i in range(num_split):
        start = 1 + i * num_columns_per_split  # Start from the second column
        end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
        split_columns = [first_column] + df.columns[start:end]
        split_df = df[split_columns]
        splitted.append(split_df)
        
    return splitted

In [ ]:
lab_split = split_df(lab,10)

In [ ]:
import math

como_med_lab = lab_split[0].join(Cohort_Commo_Med.hint('shuffle'), on='personid', how='right')
for i in range(1, len(lab_split)):
    print(i)
    como_med_lab = lab_split[i].join(como_med_lab.hint('shuffle'), on='personid', how='right')

In [ ]:
# Get the list of column names
column_names = como_med_lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [ ]:
como_med_lab.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoinFull')

In [4]:
Epilepsy_Cohort_SS = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoinFull")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
print(Epilepsy_Cohort_SS.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

152400


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Get the list of column names
column_names = Epilepsy_Cohort_SS.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 7659


In [2]:
#######Dr.Han- Starts from here ##########################
#######Step1#######
Control_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Control_S3")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
Control_Commo_Med.printSchema()

In [ ]:
# Get the list of column names
column_names = Control_Commo_Med.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [3]:
#######Step2#######
lab1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_pivot1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(lab1.count())

In [ ]:
lab1.printSchema()

In [ ]:
# Get the list of column names
column_names = lab1.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [4]:
#######Step3#######
import math

def split_df(df, num_split):
    total_columns = len(df.columns)
    first_column = df.columns[0]  # Get the name of the first column
    
    splitted = []
    
    num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
    for i in range(num_split):
        start = 1 + i * num_columns_per_split  # Start from the second column
        end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
        split_columns = [first_column] + df.columns[start:end]
        split_df = df[split_columns]
        splitted.append(split_df)
        
    return splitted

▸,:,


In [5]:
#######Step4#######
# Adjust the number of splits based on the size of lab1
num_splits = 15 
# Splitting lab1
lab1_split = split_df(lab1, num_splits)

▸,:,


In [ ]:
# print(lab_split[1].show(truncate=False))
# Get the list of column names
column_names = lab_split1[0].columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [6]:
#######Step5#######
# Joining the splits with Control_Commo_Med
control_como_med_lab = lab1_split[0].join(Control_Commo_Med.hint('shuffle'), on='personid', how='right')
for i in range(1, len(lab1_split)):
    print(i)
    control_como_med_lab = lab1_split[i].join(control_como_med_lab.hint('shuffle'), on='personid', how='right')

▸,:,


1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [ ]:
column_names = control_como_med_lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

In [7]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = control_como_med_lab.rdd.getNumPartitions()
control_como_med_lab_repart = control_como_med_lab.repartition(reparNum)

▸,:,


In [8]:
#######Step6#######
control_como_med_lab_repart.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoinFull2')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
Epilepsy_Control_SS = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoinFull1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Get the list of column names
column_names = Epilepsy_Control_SS.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 8348


In [9]:
print(Epilepsy_Control_SS.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1008863


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>